# Search and Adversarial AI

This project explores classical search and adversarial search techniques in maze-based environments, with a focus on pathfinding efficiency and decision-making under competition.

The project includes implementations and experimental comparisons of Dijkstra's algorithm, Greedy Best-First Search, A* with different heuristics, and Alpha-Beta search.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter
from IPython.display import HTML


class EmptyStackOfImages(Exception):
    pass


class visualization:
    def __init__(self, S, F):
        """
        Initialize the visualization object.

        Parameters
        ----------
        S : tuple
            Starting node of the search.
        F : tuple
            Goal node of the search.
        """
        self.S = S
        self.F = F
        self.images = []

    def draw_step(self, grid, frontier, expanded_nodes):
        """
        Create a visualization frame after a search step.

        Parameters
        ----------
        grid : grid
            Maze environment.
        frontier : list
            Nodes currently in the search frontier.
        expanded_nodes : list
            Nodes that have already been expanded.
        """
        image = np.zeros((grid.N, grid.N, 3), dtype=int)
        image[~grid.grid] = [0, 0, 0]
        image[grid.grid] = [255, 255, 255]

        for node in expanded_nodes:
            image[node] = [0, 0, 128]

        for node in frontier:
            image[node] = [0, 225, 0]

        image[self.S] = [50, 168, 64]
        image[self.F] = [168, 50, 50]
        self.images.append(image)

    def add_path(self, path):
        """
        Add the final path to the visualization.

        Parameters
        ----------
        path : list
            Sequence of nodes from start to goal.
        """
        for node in path[1:-1]:
            image = np.copy(self.images[-1])
            image[node] = [66, 221, 245]
            self.images.append(image)

        for _ in range(100):
            self.images.append(image)

    def create_gif(self, fps=30, repeat_delay=2000):
        """Create an animation from the stored frames."""
        if len(self.images) == 0:
            raise EmptyStackOfImages(
                "You must call 'draw_step' before creating an animation."
            )

        fig = plt.figure()
        plt.axis("off")
        ims = []

        for image in self.images:
            img = plt.imshow(image)
            ims.append([img])

        ani = animation.ArtistAnimation(
            fig,
            ims,
            interval=1000 // fps,
            blit=True,
            repeat_delay=repeat_delay,
        )
        plt.close(fig)
        return ani

    def save_gif(self, filename, fps=30):
        """Save the animation as a GIF file."""
        ani = self.create_gif(fps)
        writer = PillowWriter(fps=fps)
        ani.save(filename, writer=writer)

    def show_gif(self, fps=30, repeat_delay=2000):
        """Display the animation inline in a Jupyter notebook."""
        ani = self.create_gif(fps, repeat_delay)
        return HTML(ani.to_jshtml())

    def show_last_frame(self):
        """Display the most recently generated visualization frame."""
        if len(self.images) == 0:
            raise EmptyStackOfImages(
                "You must call 'draw_step' before displaying a frame."
            )

        plt.imshow(self.images[-1])

In [ ]:
from queue import PriorityQueue


class pathfinder:
    def __init__(self, S, F, grid, cost, heuristic, visualize=True):
        self.S = S
        self.F = F
        self.grid = grid
        self.cost = cost
        self.heuristic = heuristic
        self.no_expanded_nodes = 0

        self.visualize = visualize
        self.vis = visualization(S, F)

        self.find_path()

    def find_path(self):
        frontier = PriorityQueue()
        frontier.put((0, self.S))

        came_from = {self.S: None}
        cost_so_far = {self.S: 0}
        self.no_expanded_nodes = 0

        while not frontier.empty():
            current = frontier.get()[1]

            if current == self.F:
                break

            self.no_expanded_nodes += 1

            for neighbor in self.grid.adjacent_no_walls(current):
                new_cost = cost_so_far[current] + self.cost(current, neighbor)

                if (
                    neighbor not in cost_so_far
                    or new_cost < cost_so_far[neighbor]
                ):
                    cost_so_far[neighbor] = new_cost

                    priority = (
                        new_cost
                        + self.heuristic(neighbor, self.F)
                    )

                    frontier.put((priority, neighbor))
                    came_from[neighbor] = current

                # Collect frontier and expanded nodes for visualization
                restore = []

                while not frontier.empty():
                    node = frontier.get()
                    restore.append(node)

                for node in restore:
                    frontier.put(node)

                expanded = list(came_from.keys())

                if self.visualize:
                    self.vis.draw_step(
                        self.grid,
                        [item[1] for item in restore],
                        expanded,
                    )

        # Reconstruct path from goal to start
        path = []

        while current != self.S:
            path.append(current)

            if current not in came_from:
                self.grid.draw_map()
                raise ValueError("No valid path found.")

            current = came_from[current]

        path.append(self.S)
        path.reverse()

        self.path = path

    def get_path(self):
        return self.path

    def get_complexity(self):
        return self.no_expanded_nodes

    def get_path_length(self):
        return len(self.path)

## Maze Generation

In [ ]:
%matplotlib inline
import numpy as np
from queue import LifoQueue
import  random
import matplotlib.pyplot as plt


class Maze:
    def __init__(self, N, S, F, threshold=0.02):

      """
      N: integer that indicates the size of the NxN grid of the maze
      S: pair of integers that indicates the coordinates of the starting point (S)
      F: pair of integers that indicates the coordinates of the finish point (F)
      You can add any other parameters you want to customize maze creation (e.g. variables that
      control the creation of additional paths)
      """

      assert N > 2

      ## Make sure start and end are within the grid

      assert S < (N-1, N-1)
      assert F < (N-1, N-1)

      assert S > (0, 0)
      assert F > (0, 0)

      # Add here any additional constraints your implementation may have

      assert N % 2 == 1
      assert S[0] % 2 == 1
      assert S[1] % 2 == 1
      assert F[0] % 2 == 1
      assert F[1] % 2 == 1

      self.N = N
      self.S = S
      self.F = F

      # Keep track of the agents in the Maze
      self.agents=[]

      ## Initialize grid

      self.grid = np.zeros((N, N), dtype=bool)

      def neighbors(node, N, visited, threshold):
        """
        Returns all neighbors of a node that are either unvisited, or they are visited but
        there is a wall between the node and the neighbor and the neighbor passes a random test.
        """

        l = []
        x, y = node

        # first condition in all checks is for boundaries
        # neighbors are +-2 in x or y
        # walls are +-1

        if x > 2 and (not visited[x-2, y] or (not visited[x-1,y] and random.uniform(0,1) <= threshold)):
            l.append((x-2, y))
        

        #check the neighbor below
        if y > 2 and (not visited[x,y-2] or (not visited[x,y-1] and random.uniform(0,1) <= threshold)):
            l.append((x,y-2))
        #check the neighbor on the right
        if x+2 < N-1 and (not visited[x+2,y] or (not visited[x+1,y] and random.uniform(0,1) <= threshold) ):
            l.append((x+2,y))
        #check the neighbor above
        if y+2 < N-1 and (not visited[x,y+2] or (not visited[x,y+1] and random.uniform(0,1) <= threshold) ):
            l.append((x,y+2))


        
        return l

      stack = []
      stack.append(self.S)
      self.grid[self.S] = True

      while stack:
          current_node = stack.pop()
          # get all unvisited neighbors (and some visited ones with a random chance)
          n = neighbors(current_node, self.N, self.grid, threshold)
          if len(n):
              stack.append(current_node)

              
              # select a random neighbor
              next_node = random.choice(n)
              

             
              # break the wall between current and next node
              wall = (current_node[0]+(next_node[0]-current_node[0])//2,current_node[1]+(next_node[1]-current_node[1])//2)
              self.grid[wall] = True
              

              # mark next node as visited and add it to the stack
              self.grid[next_node] = True
              stack.append(next_node)



    def adjacent_no_walls(self, node):
      x, y = node
      ret = []
      if x - 1 > -1 and self.grid[x-1,y]:
        ret.append((x-1,y))
      if x + 1 < self.N and self.grid[x+1,y]:
        ret.append((x+1,y))
      if y - 1 > -1 and self.grid[x,y-1]:
        ret.append((x,y-1))
      if y + 1 < self.N and self.grid[x,y+1]:
        ret.append((x,y+1))
      return ret



    def draw_map(self, path=None,return_image=False):
        """
        Draws the maze as an image. Considers grid values of 0/False to represent obstacles and
        values of 1/True to represent empty cells, but this can be customized. Obstacles are painted
        black and empty cells are painted white. Starting point is painted green and finish point red.
        Optionally accepts as a parameter a path within the maze which is painted blue.
        """
        image = np.zeros((self.N, self.N, 3), dtype=int)
        image[~self.grid] = [0, 0, 0]
        image[self.grid] = [255, 255, 255]
        # Use this to treat 1/True as obstacles
        # image[self.grid] = [0, 0, 0]
        # image[~self.grid] = [255, 255, 255]

        image[self.S] = [50, 168, 64]
        image[self.F] = [168, 50, 50]
        if path:
            for n in path[1:-1]:
                image[n] = [66, 221, 245]

        if len(self.agents)>0:
          for a in self.agents:
            image[a.location]=a.color

        if return_image:
          return image
        else:
          plt.imshow(image)
          plt.xticks([])
          plt.yticks([])
          plt.show()



In [ ]:
for N, S, F in (11, (1, 3), (7, 9)), (25, (3, 7), (23, 19)), (51, (9, 3), (41, 41)):
    map = Maze(N, S, F, threshold=2/(N*np.log(N)))
    map.draw_map()

## Search Algorithms


In [ ]:
import math

## A heuristic
def euclidean(a, b):
    return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

# Add more heuristics here



def manhattan(a,b):
    return abs(a[0]-b[0])+abs(a[1]-b[1])




In [ ]:
## Create a 41x41 maze
N = 41
S = (5, 9)
F = (37, 37)

map = Maze(N, S, F)

In [ ]:
%%time
##Dijkstra
## Find and visualize the path
pf = pathfinder(S, F, map, lambda x, y: 1, lambda x, y: 0)
map.draw_map(pf.get_path())
pf.vis.add_path(pf.get_path())
pf.vis.show_gif()

In [ ]:
expanded_nodes = pf.get_complexity()
print('Number of expanded nodes:', expanded_nodes)
length = pf.get_path_length()
print('Number of nodes in the path:',length)

In [ ]:
%%time
#a_star euclidean
pf = pathfinder(S, F, map, lambda x, y: 1, euclidean)
map.draw_map(pf.get_path())
pf.vis.add_path(pf.get_path())
pf.vis.show_gif()

In [ ]:
expanded_nodes = pf.get_complexity()
print('Number of expanded nodes:', expanded_nodes)
length = pf.get_path_length()
print('Number of nodes in the path:',length)

In [ ]:
%%time
#best first euclidean
pf = pathfinder(S, F, map, lambda x, y: 0, euclidean)
map.draw_map(pf.get_path())
pf.vis.add_path(pf.get_path())
pf.vis.show_gif()

In [ ]:
expanded_nodes = pf.get_complexity()
print('Number of expanded nodes:', expanded_nodes)

In [ ]:
%%time
#best first manhattan
pf = pathfinder(S, F, map, lambda x, y: 0, manhattan)
map.draw_map(pf.get_path())
pf.vis.add_path(pf.get_path())
pf.vis.show_gif()

In [ ]:
expanded_nodes = pf.get_complexity()
print('Number of expanded nodes:', expanded_nodes)
length = pf.get_path_length()
print('Number of nodes in the path:',length)

## Algorithm Benchmarking

The search strategies are evaluated across randomly generated mazes of increasing size, comparing path length and the number of expanded nodes.

In [ ]:
path_lengths_a = [1,2,3,4,5,6,7,8,9]
path_lengths_b = [1,1,2,3,5,8,13,21,34]
map_sizes = [10,20,30,40,50,60,70,80,90]

plt.plot(map_sizes, path_lengths_a)
plt.plot(map_sizes, path_lengths_b )
plt.legend(['algorithm A', 'algorithm B'])
plt.title('Path length vs map size')
plt.show()

In [ ]:
S = (1, 1)

# Number of algorithms to be tested
num = 5

# Number of different maze sizes
sizes = 10

# Number of test cases for each maze size
# Temporarily set to 5 for testing; use 100 for the final benchmark
cases = 5

len_avg = []
complexity_avg = []

for i in range(sizes):
    len_sum = [0] * num
    complexity_sum = [0] * num

    for j in range(cases):
        N = 10 * (i + 1) + 1
        F = (N - 2, N - 2)
        map = Maze(N, S, F)

        # Dijkstra
        pf = pathfinder(
            S, F, map,
            lambda x, y: 1,
            lambda x, y: 0,
            visualize=False
        )
        len_sum[0] += pf.get_path_length()
        complexity_sum[0] += pf.get_complexity()

        # Best-First Search - Euclidean
        pf = pathfinder(
            S, F, map,
            lambda x, y: 0,
            euclidean,
            visualize=False
        )
        len_sum[1] += pf.get_path_length()
        complexity_sum[1] += pf.get_complexity()

        # Best-First Search - Manhattan
        pf = pathfinder(
            S, F, map,
            lambda x, y: 0,
            manhattan,
            visualize=False
        )
        len_sum[2] += pf.get_path_length()
        complexity_sum[2] += pf.get_complexity()

        # A* - Euclidean
        pf = pathfinder(
            S, F, map,
            lambda x, y: 1,
            euclidean,
            visualize=False
        )
        len_sum[3] += pf.get_path_length()
        complexity_sum[3] += pf.get_complexity()

        # A* - Manhattan
        pf = pathfinder(
            S, F, map,
            lambda x, y: 1,
            manhattan,
            visualize=False
        )
        len_sum[4] += pf.get_path_length()
        complexity_sum[4] += pf.get_complexity()

    len_avg.append([x / cases for x in len_sum])
    complexity_avg.append([x / cases for x in complexity_sum])

In [ ]:
#PLOT PATH LENGTH
#print(len_avg)
map_sizes = [10*(x+1) for x in range(sizes)]
for i in range(num):
    plt.plot(map_sizes,[row[i] for row in len_avg])
plt.legend(['Dijkstra','Best First Euclidean','Best First Manhattan','A_star Euclidean','A_star Manhattan'])
plt.title("Path length vs map size")
plt.show()

In [ ]:
#PLOT COMPLEXITIES
map_sizes = [10*(x+1) for x in range(sizes)]
for i in range(num):
    plt.plot(map_sizes,[row[i] for row in complexity_avg])
plt.legend(['Dijkstra','Best First Euclidean','Best First Manhattan','A_star Euclidean','A_star Manhattan'])
plt.title("Complexity (no of expanded nodes) vs map size")
plt.show()

## Adversarial Search

The opponent uses A* search to continuously pursue the agent based on its current position.

In [ ]:
class Agent:
  def __init__(self,S,grid,color, name = ""):
    self.agent_name = name
    self.location = S
    self.color=color
    self.maze = grid
    self.maze.agents.append(self)
    self.maze.grid[S]=1
    self.path=[]

  def find_path(self,F):
    c = lambda x,y: 1
    h = euclidean
    pf = pathfinder(self.location, F, self.maze, c, h, visualize=False)
    self.path = pf.path


  def move(self):
    if len(self.path)<1:
      return
    if self.location==self.path[0] and len(self.path)>1:
      self.location=self.path[1]
      self.path = self.path[2:]
    else:
      self.location=self.path[0]
      self.path=self.path[1:]

  def move_to(self, loc):
    self.location = loc

The agent uses Alpha-Beta search to select actions that move it toward the goal while avoiding the opponent.

In [ ]:
class ABagent:
  def __init__(self, S, grid, color, name = ""):
    self.agent_name = name
    self.location = S
    self.color=color
    self.maze = grid
    self.maze.agents.append(self)
    self.maze.grid[S]=1
    self.path=[]

  def get_best_action(self, ghosts, depth = 3):
      best_action = None
      best_score = float("-inf") #you can use this for debugging
      alpha = float("-inf")
      beta = float("inf")

      best_score, best_action = self.alpha_beta_agent(self.maze, self.location, ghosts, depth, alpha, beta, True)

      return best_action

  def alpha_beta_agent(
    self,
    maze,
    agent_pos,
    ghost_pos,
    depth,
    alpha,
    beta,
    maximizing_player=True
):
    if (
        depth == 0
        or self.is_win(maze, agent_pos)
        or self.is_lose(agent_pos, ghost_pos)
    ):
        return self.heuristic_AB(maze, agent_pos, ghost_pos), agent_pos

    if maximizing_player:
        best_score = float("-inf")
        best_action = None

        for successor_state in maze.adjacent_no_walls(agent_pos):
            score, _ = self.alpha_beta_agent(
                maze,
                successor_state,
                ghost_pos,
                depth - 1,
                alpha,
                beta,
                False
            )

            if score > best_score:
                best_score = score
                best_action = successor_state

            alpha = max(alpha, score)

            if beta <= alpha:
                break

        return best_score, best_action

    else:
        best_score = float("inf")
        best_action = None

        for successor_state in maze.adjacent_no_walls(ghost_pos):
            score, _ = self.alpha_beta_agent(
                maze,
                agent_pos,
                successor_state,
                depth - 1,
                alpha,
                beta,
                True
            )

            if score < best_score:
                best_score = score
                best_action = successor_state

            beta = min(beta, score)

            if beta <= alpha:
                break

        return best_score, best_action

          


  def is_win(self, maze, agent):
      
      return agent==maze.F
      

  def is_lose(self, agent, ghosts):
      
      return agent==ghosts
      

  def heuristic_AB(self, maze, agent_pos, ghost_pos):
    """
    Evaluate the agent's position by balancing progress toward the goal
    with maintaining distance from the ghost.
    """
    distance_to_goal = manhattan(agent_pos, maze.F)
    distance_to_ghost = manhattan(agent_pos, ghost_pos)

    return distance_to_ghost - distance_to_goal

      

  def move_to(self, loc):
    self.location = loc

A sparse maze environment is generated and both agents are initialized.

In [ ]:
#create a sparse map with many paths, like the one below
map = Maze(33, (1,1), (31,31), threshold=0.1)
map.draw_map()

In [ ]:
N = 41
map = Maze(N, (1,1), (N-2,N-2), threshold=np.log(N)/N)
x, y = np.random.choice(range(1, N-2)), np.random.choice(range(1, N-2))
ghost = (Agent((x, y), map, [255,30,10], "ghost"))

a1 = ABagent((1,1), map,[30,10,255], "agentAB")

map.draw_map()

The adversarial simulation is executed using a search depth of 5, allowing the agent to anticipate the opponent's actions when selecting its next move.

In [ ]:
vis=visualization((1,1),(31,30))

x=1
#map = lose_maps[x]
#ghost = (Agent(lose_gp[x], map, [255,30,10], "ghost"))
#a1 = ABagent((1,1), map,[30,10,255], "agentAB")
visited = [a1.location] #HINT: this might be helpful to you

for i in range(100):
  if a1.location==(31,31):
    print("Win")
    break

  if a1.location == ghost.location:
    print("Lose")
    break

  if i%1==0: #controls agent speed
    best_move = a1.get_best_action(ghost.location, 5)
    a1.move_to(best_move)
    visited.append(a1.location)

  if i%1==0: #controls ghost speed
    ghost.find_path(a1.location)
    ghost.move()

  im = map.draw_map(return_image=True)
  vis.images.append(im)

In [ ]:
vis.create_gif(fps=1)
vis.show_gif()

In [ ]:
N = 31
wins = 0
loses = 0
lose_maps = []
lose_gp = []

for testCases in range(50):

  map = Maze(N, (1,1), (N-2,N-2), threshold=np.log(N)/N)
  x, y = np.random.choice(range(1, N-2)), np.random.choice(range(1, N-2))
  ghost = (Agent((x, y), map, [255,30,10], "ghost"))
  a1 = ABagent((1,1), map,[30,10,255], "agentAB")

  for i in range(int(2*map.N*np.log(map.N))):
    if a1.location==map.F:
      wins += 1
      break

    if a1.location == ghost.location:
      loses += 1
      ghost = (Agent((x, y), map, [255,30,10], "ghost"))
      #a1 = ABagent((1,1), map,[30,10,255], "agentAB")
      lose_maps.append(map)
      lose_gp.append((x,y))
      break

    if i%1==0: #controls agent speed
      best_move = a1.get_best_action(ghost.location, 5)
      #print(i,best_move)
      a1.move_to(best_move)

    if i%1==0: #controls ghost speed
      ghost.find_path(a1.location)
      ghost.move()

print("wins = ",wins)
print("loses = ",loses)
for i in range(len(lose_maps)):
  lose_maps[i].draw_map()
  print(lose_gp[i])